# COde 00 — Research asset bank v4

Chạy **có Internet**, trên Linux/Kaggle cùng Python/scientific stack với GPU đích. Gắn source v4.
Bank cũ thiếu DINOv3/BGE-M3 nên không đủ. Nếu accelerator chặn Internet, dùng phiên free có Internet để chuẩn bị.

Trước khi tải: chấp nhận điều khoản ở Hugging Face cho facebook/dinov3-vitb16-pretrain-lvd1689m và
facebook/dinov3-vitl16-pretrain-lvd1689m. Thêm read token vào Kaggle Secrets tên HF_TOKEN, cấp quyền cho notebook.
Không dán token vào code/output/GitHub. BGE-M3: BAAI/bge-m3. Pin revision thực tế, không thay bằng random weights.

In [ ]:
from pathlib import Path
import os, sys, json, importlib.util
MODE = 'prepare'  # inspect để xem môi trường GPU đích; prepare để tải online
PROJECT_ROOT_OVERRIDE = None
BANK_ROOT = Path('/kaggle/working/code_research_bank_v1')
RUN_MINI_REHEARSAL = True
MINI_ROOT_OVERRIDE = None

def find(predicate):
    found = set()
    for base in (Path.cwd(), Path('/kaggle/input')):
        if not base.exists(): continue
        for current, dirs, files in os.walk(base):
            p = Path(current)
            if predicate(p, dirs, files): found.add(p.resolve())
            dirs[:] = [d for d in dirs if d not in {'Images','.git','.cache','_support','models','wheels','_archive'}]
            if len(p.relative_to(base).parts) >= 8: dirs[:] = []
    return sorted(found, key=str)
def one(override, found, name):
    if override: return Path(override).resolve()
    if len(found) != 1: raise RuntimeError(f'{name}: {found}. Set *_OVERRIDE.')
    return found[0]
source = one(PROJECT_ROOT_OVERRIDE, find(lambda p,d,f:
    (p / 'clindistill/research').is_dir() and 'configs' in d and p != (BANK_ROOT / 'source').resolve()), 'Source v4')
spec = importlib.util.spec_from_file_location('research_bank', source / '_support/research_bank.py')
bank = importlib.util.module_from_spec(spec); spec.loader.exec_module(bank)
print(json.dumps(bank.inventory(), indent=2))
if MODE == 'prepare' and RUN_MINI_REHEARSAL:
    if MINI_ROOT_OVERRIDE:
        mini = Path(MINI_ROOT_OVERRIDE)
    else:
        full_release = one(None, find(lambda p,d,f: 'release.json' in f and
            not bank.read_json(p / 'release.json').get('debug_subset')), 'Full prepared COde release')
        full_images = one(None, find(lambda p,d,f: (p / 'Images/Photographs').is_dir() and
            'mini_dataset.json' not in f), 'Full COde images')
        mini = BANK_ROOT.parent / 'research_mini_v1'
        mini_spec = importlib.util.spec_from_file_location('mini_builder', source / '_support/kaggle_offline.py')
        mini_builder = importlib.util.module_from_spec(mini_spec); mini_spec.loader.exec_module(mini_builder)
        if not (mini / 'mini_dataset.json').exists():
            import uuid
            staging = mini.with_name(mini.name + '_staging_' + uuid.uuid4().hex[:8])
            mini_builder.make_mini_code(full_release, full_images, staging, per_class=2, independent_groups=True)
            staging.rename(mini)
        elif bank.read_json(mini / 'prepared/release.json')['source_release_id'] != bank.read_json(full_release / 'release.json')['release_id']:
            raise RuntimeError('Mini thuộc release khác. Đặt MINI_ROOT_OVERRIDE tới mini đúng release.')
if MODE == 'prepare':
    if not os.environ.get('HF_TOKEN'):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            raise RuntimeError('Thêm Kaggle Secret HF_TOKEN và cấp quyền trước khi prepare.') from None
    info = bank.prepare(source, BANK_ROOT)
elif MODE != 'inspect': raise ValueError('MODE must be inspect or prepare.')

## Thử pipeline thật trên mini COde

Gắn code-training-inputs đầy đủ: cell trên tự trích mini 2 nhóm độc lập/lớp từ prepared release,
không mở test. Mini cũ có thể có nhiều ca cùng bệnh nhân nên chưa đủ 2-fold OOF.
2-fold, 1 seed, 3 epochs kiểm tra phần mềm với pretrained thật; không dùng số đo này trong khóa luận.
Bank được gắn trực tiếp từ Output sang notebook01/02/03, không cần tải về máy.

In [ ]:
if MODE == 'prepare' and RUN_MINI_REHEARSAL:
    sys.path.insert(0, str(BANK_ROOT / 'source'))
    from clindistill.artifacts import read_json, write_json
    from clindistill.research.data import prepare_features
    from clindistill.research.training import run_study
    from clindistill.research.training import grouped_folds
    rows = [json.loads(line) for line in (mini / 'prepared/student_single_train.jsonl').read_text(encoding='utf-8').splitlines() if line]
    grouped_folds([r['single_label_index'] for r in rows], [r['split_group'] for r in rows], 2, 2026)
    models = {k: BANK_ROOT / v for k,v in info['models'].items()}
    cfg = read_json(BANK_ROOT / 'source/configs/code/research.json')
    cfg.update(debug=True, seeds=[2026], folds=2, epochs=3, patience=3, feature_dim=64,
        learning_rate=0.01, bootstrap_repetitions=10,
        methods=['student_no_kd','kd_visual','kd_multimodal','kd_confidence','kd_reliability'])
    cache = BANK_ROOT.parent / 'research_mini_features'
    with bank.block_network():
        prepare_features(mini / 'prepared', mini, models, cache, cfg)
        display(run_study(cache, BANK_ROOT.parent / 'research_mini_runs', cfg, require_gpu=False))
    write_json(BANK_ROOT / 'rehearsal.json', {'status':'passed','debug':True,'contains_test':False})
print('Save Version. 01/02/03 Add Input > Notebook Output > chọn output có research_bank_manifest.json.')

Bank gồm source, Linux wheels, DINO-B/L, BGE-M3, ResNet/Swin weights; không gồm COde đầy đủ.
Giữ bank, mini, cache và checkpoints riêng tư. Nếu chưa có quyền DINO, dừng ở tải model và hoàn tất quyền truy cập;
không có cơ chế thay model hoặc sinh kết quả giả. Resume prepare dùng lại model downloads đã có.